In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_starcraft")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_players = spark.table(f"{catalogo}.{esquema_source}.players_stats")

In [0]:
# total de juegos, total de victorias y total de derrotas por jugador, la columna win_rate debe tener 2 decimales

df_players_stats = df_players.withColumn('total_games', F.expr("vs_zerg_win + vs_zerg_los + vs_protoss_win + vs_protoss_los + vs_terran_win + vs_terran_los")) \
                             .withColumn('total_wins', F.expr("vs_zerg_win + vs_protoss_win + vs_terran_win")) \
                             .withColumn('total_loses', F.expr("vs_zerg_los + vs_protoss_los + vs_terran_los")) \
                             .withColumn('win_rate', F.round(F.expr("total_wins / total_games"), 2))


In [0]:
display(df_players_stats)

In [0]:
df_players_stats.printSchema()

### Guardar Tabla Golden_Players_Stats

In [0]:
df_players_stats.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{esquema_sink}.golden_players_stats')

In [0]:
# grafico de barras con el win_rate por jugador
import plotly.express as px

fig = px.bar(df_players_stats.toPandas(), x='player', y='win_rate', color='player', title='Win Rate por Jugador')
fig.show()